# llm_gateway tests\n\nRun cells top-to-bottom.\n- Offline smoke tests: local code behavior only.\n- Online API pings: direct calls to provider endpoints (no localhost gateway needed).

In [ ]:
from __future__ import annotations\n\nimport os\nimport sys\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nROOT = Path.cwd()\nLLM_GATEWAY_DIR = ROOT / "llm_gateway"\nif str(LLM_GATEWAY_DIR) not in sys.path:\n    sys.path.insert(0, str(LLM_GATEWAY_DIR))\n\nprint(f"Using llm_gateway path: {LLM_GATEWAY_DIR}")\nprint(f"Path exists: {LLM_GATEWAY_DIR.exists()}")

In [ ]:
from gateway.api.schemas import DispatchRequest\nfrom gateway.config.settings import settings\nfrom gateway.providers.base import ProviderAdapterError\nfrom gateway.providers.registry import PROVIDERS\n\nfrom gateway.providers.openrouter import OpenRouterAdapter\nfrom gateway.providers.cohere import CohereAdapter\nfrom gateway.providers.huggingface import HuggingFaceAdapter\nfrom gateway.providers.gemini import GeminiAdapter\n\n\ndef run_test(name, fn):\n    try:\n        fn()\n        print(f"[PASS] {name}")\n        return True\n    except Exception as exc:\n        print(f"[FAIL] {name}: {type(exc).__name__}: {exc}")\n        return False\n\n\ndef test_provider_registry():\n    expected = {"openrouter", "gemini", "cohere", "huggingface"}\n    assert expected.issubset(set(PROVIDERS)), f"Missing providers: {expected - set(PROVIDERS)}"\n\n\ndef test_settings_loaded():\n    assert isinstance(settings.default_max_output_tokens, int)\n    assert settings.default_max_output_tokens > 0\n    assert isinstance(settings.allow_paid_fallback, bool)\n\n\ndef test_schema_validation():\n    req = DispatchRequest(messages=[{"role": "user", "content": "hello"}], require_json=True)\n    assert req.task_type == "chat"\n    assert req.messages[0].role == "user"\n    assert req.require_json is True\n\n\ndef test_payload_builders():\n    req = DispatchRequest(\n        messages=[{"role": "user", "content": "Give me 2 bullet points about convexity."}],\n        max_output_tokens=80,\n        require_json=False,\n    )\n    route = SimpleNamespace(model_name="dummy-model", supports_json=True)\n\n    for adapter in [OpenRouterAdapter(), CohereAdapter(), HuggingFaceAdapter()]:\n        payload = adapter.build_payload(req, route)\n        assert "messages" in payload.payload\n        assert payload.token_estimate.input_tokens is not None\n        assert payload.token_estimate.output_tokens == 80\n\n\ndef test_gemini_missing_key_behavior():\n    req = DispatchRequest(messages=[{"role": "user", "content": "hello"}])\n    route = SimpleNamespace(model_name="gemini-2.5-flash", supports_json=True)\n    try:\n        GeminiAdapter().build_payload(req, route)\n    except ProviderAdapterError as exc:\n        assert exc.error_code == "missing_key"\n        return\n    assert os.getenv("GEMINI_API_KEY") or settings.gemini_api_key\n\n\ntests = [\n    ("provider registry", test_provider_registry),\n    ("settings load", test_settings_loaded),\n    ("schema validation", test_schema_validation),\n    ("payload builders", test_payload_builders),\n    ("gemini missing-key behavior", test_gemini_missing_key_behavior),\n]\n\nresults = [run_test(name, fn) for name, fn in tests]\nprint(f"\nSummary: {sum(results)}/{len(results)} tests passed")

In [ ]:
# Key visibility check (no secret values printed)\nkey_names = [\n    "GEMINI_API_KEY",\n    "COHERE_API_KEY",\n    "HF_API_KEY",\n    "OPENROUTER_API_KEY",\n]\nfor k in key_names:\n    print(f"{k}: {'SET' if os.getenv(k) else 'missing'}")\n\nprint("\nNote: gateway also loads keys from llm_gateway/.env via pydantic settings.")

In [ ]:
# Optional direct online API pings (no localhost).\n# These are lightweight auth/connectivity checks to provider APIs.\n\nimport json\nimport httpx\n\nresults = {}\n\n\ndef mark(provider, ok, detail):\n    results[provider] = {"ok": ok, "detail": detail}\n    tag = "PASS" if ok else "FAIL"\n    print(f"[{tag}] {provider}: {detail}")\n\n\ndef safe_json_snippet(resp):\n    try:\n        return json.dumps(resp.json(), indent=2)[:300]\n    except Exception:\n        return (resp.text or "")[:300]\n\n\n# OpenRouter ping\nopenrouter_key = os.getenv("OPENROUTER_API_KEY")\nif not openrouter_key:\n    mark("openrouter", False, "OPENROUTER_API_KEY missing")\nelse:\n    headers = {"Authorization": f"Bearer {openrouter_key}"}\n    site = os.getenv("OPENROUTER_SITE_URL")\n    app_name = os.getenv("OPENROUTER_APP_NAME", "llm-gateway-tests")\n    if site:\n        headers["HTTP-Referer"] = site\n    if app_name:\n        headers["X-Title"] = app_name\n    try:\n        resp = httpx.get("https://openrouter.ai/api/v1/key", headers=headers, timeout=20.0)\n        mark("openrouter", resp.status_code == 200, f"HTTP {resp.status_code} | {safe_json_snippet(resp)}")\n    except Exception as exc:\n        mark("openrouter", False, f"{type(exc).__name__}: {exc}")\n\n\n# Cohere ping\ncohere_key = os.getenv("COHERE_API_KEY")\nif not cohere_key:\n    mark("cohere", False, "COHERE_API_KEY missing")\nelse:\n    headers = {"Authorization": f"Bearer {cohere_key}"}\n    try:\n        resp = httpx.get("https://api.cohere.com/v1/models", headers=headers, timeout=20.0)\n        ok = resp.status_code < 400\n        mark("cohere", ok, f"HTTP {resp.status_code} | {safe_json_snippet(resp)}")\n    except Exception as exc:\n        mark("cohere", False, f"{type(exc).__name__}: {exc}")\n\n\n# Hugging Face Router ping\nhf_key = os.getenv("HF_API_KEY")\nif not hf_key:\n    mark("huggingface", False, "HF_API_KEY missing")\nelse:\n    headers = {"Authorization": f"Bearer {hf_key}"}\n    try:\n        resp = httpx.get("https://router.huggingface.co/v1/models", headers=headers, timeout=20.0)\n        ok = resp.status_code < 400\n        mark("huggingface", ok, f"HTTP {resp.status_code} | {safe_json_snippet(resp)}")\n    except Exception as exc:\n        mark("huggingface", False, f"{type(exc).__name__}: {exc}")\n\n\n# Gemini ping\ngemini_key = os.getenv("GEMINI_API_KEY")\nif not gemini_key:\n    mark("gemini", False, "GEMINI_API_KEY missing")\nelse:\n    try:\n        from google import genai\n\n        model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")\n        client = genai.Client(api_key=gemini_key)\n        counted = client.models.count_tokens(model=model, contents="ping")\n        total = getattr(counted, "total_tokens", None)\n        mark("gemini", True, f"count_tokens ok | model={model} | total_tokens={total}")\n    except Exception as exc:\n        mark("gemini", False, f"{type(exc).__name__}: {exc}")\n\n\npassed = sum(1 for v in results.values() if v["ok"])\nprint(f"\nOnline ping summary: {passed}/{len(results)} passed")